# U²-Net-p — Background Removal

The ~4.5 MB salient-object segmentation network behind Xovato's background removal. Real inference, real timing, real alpha.

**Model:** `u2netp.onnx`  |  **Feature:** Background removal (salient object segmentation)

Runs on CPU or GPU. Cells: setup → data → load → test → benchmark → accuracy → training notes → copyright.


## 1. Setup - code + dependencies


In [ ]:
import os, sys, subprocess, time, io, urllib.request, platform
from pathlib import Path
REPO = 'https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git'
if Path('app').exists(): ROOT = Path('.').resolve()
elif Path('../app').exists(): ROOT = Path('..').resolve()
else:
    subprocess.run(['git','clone','--depth','1',REPO], check=True)
    ROOT = Path('Xovato-cpu-ai-studio').resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','pillow','numpy','onnxruntime','soundfile'], check=False)
print('root:', ROOT, '| python', platform.python_version())

## 2. Data (test set)


In [ ]:
import numpy as np
from PIL import Image, ImageDraw
rng = np.random.default_rng(0)
def subject(w=600,h=600):
    img = Image.fromarray(rng.integers(80,200,(h,w,3)).astype('uint8')); d=ImageDraw.Draw(img)
    d.ellipse([250,120,350,220], fill=(235,205,180)); d.rectangle([230,220,370,500], fill=(200,70,70))
    # ground-truth mask for the subject (for the IoU accuracy signal)
    gt = Image.new('L',(w,h),0); dg=ImageDraw.Draw(gt)
    dg.ellipse([250,120,350,220], fill=255); dg.rectangle([230,220,370,500], fill=255)
    return img, np.asarray(gt)>127
img, gt_mask = subject(); img

## 3. Download & load the model


In [ ]:
def fetch(url, dest, min_bytes):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print('present', dest.name, f'{dest.stat().st_size/1e6:.1f}MB'); return dest
    print('downloading', dest.name, '...'); urllib.request.urlretrieve(url, dest)
    assert dest.stat().st_size >= min_bytes, 'file too small (LFS pointer?)'
    print('ok', f'{dest.stat().st_size/1e6:.1f}MB'); return dest

u2 = fetch('https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2netp.onnx',
           ROOT/'models'/'segmentation'/'u2netp.onnx', 3_000_000)

## 4. Inference test


In [ ]:
from app import image_processing as ip
buf = io.BytesIO(); img.save(buf, format='JPEG')
payload, analysis = ip.remove_background(buf.getvalue(), model_path=u2, output_format='png')
print(analysis.summary); Image.open(io.BytesIO(payload))

## 5. Benchmark (latency)


In [ ]:
N=5; buf=io.BytesIO(); img.save(buf,format='JPEG'); data=buf.getvalue()
t=time.perf_counter()
for _ in range(N): ip.remove_background(data, model_path=u2, output_format='png')
print(f'{(time.perf_counter()-t)/N*1000:.0f} ms/image over {N} runs')

## 6. Accuracy signal


In [ ]:
# IoU of the predicted alpha vs the known subject mask on our test image.
from app.image_processing import _predict_mask
mask = _predict_mask(img.convert('RGB'), u2)
pred = np.asarray(Image.fromarray((mask*255).astype('uint8')).resize(img.size))>127
inter=(pred&gt_mask).sum(); union=(pred|gt_mask).sum()
print(f'IoU on test subject: {inter/max(union,1):.3f}')
print('Reported (paper, ECSSD): maxF-beta ~0.911, MAE ~0.041 - run the real set below.')
# Full benchmark: download ECSSD/DUTS and compute maxFbeta/MAE per the U^2-Net paper.

## 7. Training / fine-tuning notes

U²-Net was trained on **DUTS-TR** (10,553 images) for salient object detection (Qin et al., *Pattern Recognition* 2020). To fine-tune on your own cut-outs, prepare `(image, binary-mask)` pairs and train the PyTorch U²-Net, then export to ONNX with `torch.onnx.export`. This notebook uses the released ONNX weights for inference; a full training run needs a GPU and is out of scope for a CPU demo.


## 8. Copyright


In [ ]:
# ====================================================================
# COPYRIGHT NOTICE - DO NOT REMOVE
# ====================================================================
# (c) Zubair Hussain - Xovato AI Studio.
# These notebooks, the benchmark/accuracy harness, and the surrounding
# tooling are the work of Zubair Hussain. You may use and modify them for
# your own projects; keep this attribution intact, including in the
# background-removal notebook and any derived outputs.
print('(c) Zubair Hussain - Xovato AI Studio')